# Rule-based Agent — Caro 11×11

**Tác giả:** Lê Tấn Phát  
**Phạm vi:** Giai đoạn 1 — Luật thắng ngay và chặn thua (Phần 1–4 hoàn chỉnh; Phần 5–10 sẽ hoàn thiện ở Giai đoạn 2 & 3)  
**Nguồn module:** `agents/rule_based_agent.py`  
**Lưu ý:** Mọi lần chạy nên bắt đầu bằng **Restart Kernel & Run All** để đảm bảo tính tái lập.


## Phần 1 — Câu hỏi nghiên cứu và phạm vi

**Câu hỏi trung tâm:**  
Việc bổ sung nhận diện pattern nâng cao (open-four, open-three, chuỗi có khoảng trống) có thực sự nâng cao chất lượng nước đi so với việc chỉ dùng luật thắng ngay/chặn thua kết hợp điểm vị trí cơ bản hay không?

**Phạm vi Phần 1–4 (Giai đoạn 1):**
- Xây dựng baseline chiến thuật: thắng ngay → chặn thua → chọn vị trí tốt.
- Kiểm chứng thực nghiệm cho thấy luật thắng/chặn hoạt động đúng ở cả 4 hướng và 2 phía.
- Chưa cài đặt pattern nâng cao (`use_patterns=False` ở Giai đoạn 1).

**Hạn chế đã biết ở Giai đoạn 1:**
- Nước dự phòng (fallback) chỉ dựa vào điểm vị trí đơn giản, chưa nhận diện open-four hay open-three.
- Không xử lý đe dọa kép (double threat) của đối thủ — đây là giới hạn tự nhiên của tầng luật hiện tại.
- Tập ứng viên fallback bán kính 2 có thể bỏ sót các vị trí chiến lược xa quân cờ hiện có.


## Phần 2 — Môi trường và cấu hình

In [1]:
from pathlib import Path
import os

# Chuyển về thư mục caro-11x11 để import module chung từ vị trí notebook.
if Path.cwd().name == 'notebooks' and (Path.cwd().parent / 'core').is_dir():
    os.chdir(Path.cwd().parent)
if not (Path.cwd() / 'core' / 'game_engine.py').is_file():
    raise RuntimeError('Hãy mở notebook từ thư mục caro-11x11 hoặc notebooks')

print(f'Thư mục làm việc: {Path.cwd()}')

Thư mục làm việc: D:\TTNTNC\AdvancedAI-Repo\caro-11x11


In [2]:
import numpy as np

from core import CaroGame
from core.game_utils import candidate_actions
from agents import RandomAgent

# Import trực tiếp từ module rule_based_agent (không chép lại thuật toán)
from agents.rule_based_agent import (
    RuleBasedAgent,
    is_winning_move,
    find_winning_actions,
    positional_score,
    DIRECTIONS,
)
from config.game_config import BOARD_SIZE, WIN_LENGTH, PLAYER_X, PLAYER_O, EMPTY

print('Import thành công.')

Import thành công.


In [3]:
# ---- Cấu hình thí nghiệm ----
CANDIDATE_RADIUS = 2       # Bán kính tập ứng viên cho fallback
TIEBREAK_SEED = None       # None = tất định; đặt số nguyên để ngẫu nhiên có kiểm soát
GAME_SEED = 42             # Seed cho RandomAgent trong thi đấu thử nghiệm
WIN_LENGTH_CFG = WIN_LENGTH  # 5 quân liên tiếp

# Khởi tạo agent baseline (Giai đoạn 1: use_patterns=False)
agent = RuleBasedAgent(
    name='RuleBased-Baseline',
    candidate_radius=CANDIDATE_RADIUS,
    use_patterns=False,
    seed=TIEBREAK_SEED,
)

print(f'Tác tử: {agent.name}')
print(f'Bán kính ứng viên: {agent.candidate_radius}')
print(f'use_patterns: {agent.use_patterns}')
print(f'Seed tie-breaking: {agent.seed}')
print(f'Kích thước bàn cờ: {BOARD_SIZE}×{BOARD_SIZE}')
print(f'Độ dài thắng: {WIN_LENGTH_CFG} quân liên tiếp')

Tác tử: RuleBased-Baseline
Bán kính ứng viên: 2
use_patterns: False
Seed tie-breaking: None
Kích thước bàn cờ: 11×11
Độ dài thắng: 5 quân liên tiếp


### Thuyết minh cấu hình và cách đo

**1. Công thức `positional_score(board, row, col, player)`**

Điểm gần tâm theo khoảng cách Chebyshev (với `center = BOARD_SIZE // 2 = 5`):

$$score_{tam} = center - \max(|r - center|,\ |c - center|)$$

Cộng thêm điểm thưởng lân cận (duyệt các ô trong cửa sổ bán kính 2 quanh `(row, col)`):

| Loại quân lân cận | Khoảng cách | Điểm cộng |
| :--- | :---: | :---: |
| Đồng minh (cùng màu `player`) | bán kính 1 | $+0.3$ |
| Đồng minh (cùng màu `player`) | bán kính 2 | $+0.1$ |
| Đối thủ | bán kính 2 (gồm cả bán kính 1) | $+0.05$ |

**2. Quy tắc phá hòa (tie-breaking)**

- `seed=None` (tất định): chọn nước có khóa lớn nhất theo `(positional_score, -manhattan_dist_to_center, -row, -col)` — tức điểm vị trí cao nhất, rồi khoảng cách Manhattan tới tâm nhỏ nhất, rồi tọa độ `(row, col)` nhỏ nhất.
- `seed != None`: dùng `np.random.default_rng(seed)` để chọn ngẫu nhiên có kiểm soát trong nhóm `top_group` gồm các nước có `positional_score` cao nhất bằng nhau.

**3. Phương pháp đếm `stats["nodes_visited"]`** (đặt lại về 0 ở đầu mỗi lượt đi)

- Tầng `win_now`: mỗi ô trống được kiểm tra tính là 1 node, cộng `+total_empty`.
- Tầng `block_urgent` (chỉ chạy nếu không có nước thắng ngay): quét thêm toàn bộ ô trống để tìm đe dọa của đối thủ, cộng `+total_empty`.
- Tầng `positional_fallback` (chỉ chạy nếu không có nước thắng và không phải chặn): cộng thêm số ứng viên trong bán kính `candidate_radius`, tức `+len(candidates)`.

## Phần 3 — Bộ vị trí chiến thuật

Dưới đây là các tình huống thử nghiệm được dựng thông qua chuỗi nước đi hợp lệ trên `CaroGame.step()` (đảm bảo cân bằng số quân theo đúng lượt đi: $|X| = |O|$ khi đến lượt X, $|X| = |O| + 1$ khi đến lượt O), kết hợp hàm tạo mảng trực tiếp cho các ca kiểm thử biên đặc biệt. Mỗi tình huống nêu rõ:
- Trạng thái bàn cờ
- Bên đến lượt
- Danh sách nước đi chấp nhận được và lý do

In [4]:
def make_board(x_stones, o_stones):
    """Dựng bàn cờ 11×11 bằng chuỗi nước đi hợp lệ qua CaroGame.step().

    X đi trước, hai bên xen kẽ. Số quân phải hợp lệ theo lượt đi:
      - len(x_stones) == len(o_stones)     -> đến lượt X
      - len(x_stones) == len(o_stones) + 1 -> đến lượt O
    Trả về bản sao của board (không giữ tham chiếu tới engine).
    """
    assert len(x_stones) - len(o_stones) in (0, 1), (
        f'Số quân không hợp lệ theo lượt đi: |X|={len(x_stones)}, |O|={len(o_stones)}'
    )
    game = CaroGame()
    for i, x_move in enumerate(x_stones):
        game.step(x_move)
        if i < len(o_stones):
            game.step(o_stones[i])
    return game.board.copy()

def show_board(board, title=''):
    """Hiển thị bàn cờ dạng text."""
    symbols = {0: '.', 1: 'X', -1: 'O'}
    if title:
        print(f'=== {title} ===')
    header = '   ' + ' '.join(f'{c:2}' for c in range(board.shape[1]))
    print(header)
    for r, row in enumerate(board):
        line = f'{r:2} ' + ' '.join(f'{symbols[int(v)]:2}' for v in row)
        print(line)
    print()

# Tình huống 1: X sắp thắng ngang (đến lượt X: |X| = |O| = 4)
board_win_h = make_board(
    x_stones=[(5, 1), (5, 2), (5, 3), (5, 4)],
    o_stones=[(0, 0), (0, 2), (0, 4), (0, 6)],
)
show_board(board_win_h, 'Tình huống 1: X có 4 quân ngang hàng 5')
print(f'Nước đi chấp nhận được (X): {find_winning_actions(board_win_h, PLAYER_X)}')
print('Lý do: Bất kỳ ô nào trong tập trên đều tạo chuỗi 5 quân → X thắng ngay.')
print()

=== Tình huống 1: X có 4 quân ngang hàng 5 ===
    0  1  2  3  4  5  6  7  8  9 10
 0 O  .  O  .  O  .  O  .  .  .  . 
 1 .  .  .  .  .  .  .  .  .  .  . 
 2 .  .  .  .  .  .  .  .  .  .  . 
 3 .  .  .  .  .  .  .  .  .  .  . 
 4 .  .  .  .  .  .  .  .  .  .  . 
 5 .  X  X  X  X  .  .  .  .  .  . 
 6 .  .  .  .  .  .  .  .  .  .  . 
 7 .  .  .  .  .  .  .  .  .  .  . 
 8 .  .  .  .  .  .  .  .  .  .  . 
 9 .  .  .  .  .  .  .  .  .  .  . 
10 .  .  .  .  .  .  .  .  .  .  . 

Nước đi chấp nhận được (X): [(5, 0), (5, 5)]
Lý do: Bất kỳ ô nào trong tập trên đều tạo chuỗi 5 quân → X thắng ngay.



In [5]:
# Tình huống 2: O đang đe dọa, X phải chặn (đến lượt X: |X| = |O| = 4)
board_block = make_board(
    x_stones=[(5, 5), (6, 6), (7, 7), (9, 1)],
    o_stones=[(0, 7), (1, 7), (2, 7), (3, 7)],
)
show_board(board_block, 'Tình huống 2: O có 4 quân dọc cột 7, X phải chặn')
o_threats = find_winning_actions(board_block, PLAYER_O)
print(f'Đe dọa của O (nước thắng ngay cho O): {o_threats}')
print(f'X không có nước thắng: {find_winning_actions(board_block, PLAYER_X)}')
print('Lý do: X phải chặn tại một trong các ô trên để O không thắng ngay.')
print()

=== Tình huống 2: O có 4 quân dọc cột 7, X phải chặn ===
    0  1  2  3  4  5  6  7  8  9 10
 0 .  .  .  .  .  .  .  O  .  .  . 
 1 .  .  .  .  .  .  .  O  .  .  . 
 2 .  .  .  .  .  .  .  O  .  .  . 
 3 .  .  .  .  .  .  .  O  .  .  . 
 4 .  .  .  .  .  .  .  .  .  .  . 
 5 .  .  .  .  .  X  .  .  .  .  . 
 6 .  .  .  .  .  .  X  .  .  .  . 
 7 .  .  .  .  .  .  .  X  .  .  . 
 8 .  .  .  .  .  .  .  .  .  .  . 
 9 .  X  .  .  .  .  .  .  .  .  . 
10 .  .  .  .  .  .  .  .  .  .  . 

Đe dọa của O (nước thắng ngay cho O): [(4, 7)]
X không có nước thắng: []
Lý do: X phải chặn tại một trong các ô trên để O không thắng ngay.



In [6]:
# Tình huống 3: Xung đột ưu tiên — X thắng ngay VÀ O đang đe dọa (đến lượt X: |X| = |O| = 4)
board_conflict = make_board(
    x_stones=[(5, 1), (5, 2), (5, 3), (5, 4)],
    o_stones=[(0, 9), (1, 9), (2, 9), (3, 9)],
)
show_board(board_conflict, 'Tình huống 3: X sắp thắng VÀ O đang đe dọa (xung đột ưu tiên)')
print(f'Nước thắng ngay của X: {find_winning_actions(board_conflict, PLAYER_X)}')
print(f'Đe dọa của O: {find_winning_actions(board_conflict, PLAYER_O)}')
print('Nước chấp nhận được: X phải chọn nước THẮNG NGAY (không được chặn O trước).')
print()

=== Tình huống 3: X sắp thắng VÀ O đang đe dọa (xung đột ưu tiên) ===
    0  1  2  3  4  5  6  7  8  9 10
 0 .  .  .  .  .  .  .  .  .  O  . 
 1 .  .  .  .  .  .  .  .  .  O  . 
 2 .  .  .  .  .  .  .  .  .  O  . 
 3 .  .  .  .  .  .  .  .  .  O  . 
 4 .  .  .  .  .  .  .  .  .  .  . 
 5 .  X  X  X  X  .  .  .  .  .  . 
 6 .  .  .  .  .  .  .  .  .  .  . 
 7 .  .  .  .  .  .  .  .  .  .  . 
 8 .  .  .  .  .  .  .  .  .  .  . 
 9 .  .  .  .  .  .  .  .  .  .  . 
10 .  .  .  .  .  .  .  .  .  .  . 

Nước thắng ngay của X: [(5, 0), (5, 5)]
Đe dọa của O: [(4, 9)]
Nước chấp nhận được: X phải chọn nước THẮNG NGAY (không được chặn O trước).



In [7]:
# Tình huống 4: Bàn cờ trống — fallback vị trí (X đi nước đầu: |X| = |O| = 0)
board_empty = make_board([], [])
show_board(board_empty, 'Tình huống 4: Bàn cờ trống — không có đe dọa trực tiếp')
print('Nước đi chấp nhận được: (5,5) — tâm bàn cờ (điểm vị trí cao nhất).')
print('Lý do: Fallback positional_score ưu tiên tâm bàn cờ theo khoảng cách Chebyshev.')
print()

=== Tình huống 4: Bàn cờ trống — không có đe dọa trực tiếp ===
    0  1  2  3  4  5  6  7  8  9 10
 0 .  .  .  .  .  .  .  .  .  .  . 
 1 .  .  .  .  .  .  .  .  .  .  . 
 2 .  .  .  .  .  .  .  .  .  .  . 
 3 .  .  .  .  .  .  .  .  .  .  . 
 4 .  .  .  .  .  .  .  .  .  .  . 
 5 .  .  .  .  .  .  .  .  .  .  . 
 6 .  .  .  .  .  .  .  .  .  .  . 
 7 .  .  .  .  .  .  .  .  .  .  . 
 8 .  .  .  .  .  .  .  .  .  .  . 
 9 .  .  .  .  .  .  .  .  .  .  . 
10 .  .  .  .  .  .  .  .  .  .  . 

Nước đi chấp nhận được: (5,5) — tâm bàn cờ (điểm vị trí cao nhất).
Lý do: Fallback positional_score ưu tiên tâm bàn cờ theo khoảng cách Chebyshev.



## Phần 4 — Kiểm chứng luật thắng và chặn

Trong phần này, ta quan sát trực tiếp phản ứng của `RuleBasedAgent` trên các bộ vị trí chiến thuật ở:
- Cả 4 hướng: ngang, dọc, chéo chính, chéo phụ
- Cả 2 phía: X và O
- Vị trí sát biên bàn cờ
- Chuỗi dài hơn 5 quân
- Đồng thời có cơ hội thắng lẫn đe dọa thua

In [8]:
def test_scenario(board, player, expected_rule, acceptable_actions=None, title=''):
    """Chạy agent trên một tình huống và in kết quả quan sát."""
    _agent = RuleBasedAgent(seed=None)  # tất định
    board_copy = board.copy()
    action = _agent.choose_action(board_copy, player)
    info = _agent.explain_last_action()
    
    # Kiểm tra bất biến board
    board_unchanged = np.array_equal(board, board_copy)
    rule_ok = info['rule_triggered'] == expected_rule
    action_ok = (acceptable_actions is None) or (action in acceptable_actions)
    
    status = '✓ PASS' if (rule_ok and action_ok and board_unchanged) else '✗ FAIL'
    player_sym = 'X' if player == PLAYER_X else 'O'
    
    print(f'{status} [{title}]')
    print(f'  Bên đến lượt: {player_sym}')
    print(f'  Luật kích hoạt: {info["rule_triggered"]} (kỳ vọng: {expected_rule})')
    print(f'  Nước chọn: {action}')
    print(f'  Lý do: {info["reason"]}')
    print(f'  Board bất biến: {board_unchanged}')
    if acceptable_actions:
        print(f'  Nước đi chấp nhận được: {acceptable_actions}')
        print(f'  Nước chọn hợp lệ: {action_ok}')
    print()
    return status

print('=== PHẦN 4A: Kiểm chứng thắng ngay (win_now) — 4 hướng × 2 phía ===')
print()

# Ngang — X thắng (lượt X: 4 và 4)
b = make_board([(5, 1), (5, 2), (5, 3), (5, 4)], [(0, 0), (0, 2), (0, 4), (0, 6)])
test_scenario(b, PLAYER_X, 'win_now', find_winning_actions(b, PLAYER_X),
              'Ngang — X thắng')

# Ngang — O thắng (lượt O: 5 và 4)
b = make_board([(0, 0), (0, 2), (0, 4), (2, 0), (2, 2)], [(7, 1), (7, 2), (7, 3), (7, 4)])
test_scenario(b, PLAYER_O, 'win_now', find_winning_actions(b, PLAYER_O),
              'Ngang — O thắng')

# Dọc — X thắng (lượt X: 4 và 4)
b = make_board([(0, 5), (1, 5), (2, 5), (3, 5)], [(0, 0), (0, 2), (0, 4), (0, 6)])
test_scenario(b, PLAYER_X, 'win_now', find_winning_actions(b, PLAYER_X),
              'Dọc — X thắng')

# Dọc — O thắng (lượt O: 5 và 4)
b = make_board([(0, 0), (0, 2), (0, 4), (2, 0), (2, 2)], [(1, 3), (2, 3), (3, 3), (4, 3)])
test_scenario(b, PLAYER_O, 'win_now', find_winning_actions(b, PLAYER_O),
              'Dọc — O thắng')

# Chéo chính — X thắng (lượt X: 4 và 4)
b = make_board([(0, 0), (1, 1), (2, 2), (3, 3)], [(0, 2), (0, 4), (2, 0), (4, 0)])
test_scenario(b, PLAYER_X, 'win_now', find_winning_actions(b, PLAYER_X),
              'Chéo chính — X thắng')

# Chéo phụ — O thắng (lượt O: 5 và 4)
b = make_board([(0, 0), (0, 2), (0, 4), (2, 0), (2, 2)], [(1, 9), (2, 8), (3, 7), (4, 6)])
test_scenario(b, PLAYER_O, 'win_now', find_winning_actions(b, PLAYER_O),
              'Chéo phụ — O thắng')

=== PHẦN 4A: Kiểm chứng thắng ngay (win_now) — 4 hướng × 2 phía ===

✓ PASS [Ngang — X thắng]
  Bên đến lượt: X
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (5, 5)
  Lý do: Thắng ngay tại (5, 5) — tìm thấy 2 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(5, 0), (5, 5)]
  Nước chọn hợp lệ: True

✓ PASS [Ngang — O thắng]
  Bên đến lượt: O
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (7, 5)
  Lý do: Thắng ngay tại (7, 5) — tìm thấy 2 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(7, 0), (7, 5)]
  Nước chọn hợp lệ: True

✓ PASS [Dọc — X thắng]
  Bên đến lượt: X
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (4, 5)
  Lý do: Thắng ngay tại (4, 5) — tìm thấy 1 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(4, 5)]
  Nước chọn hợp lệ: True

✓ PASS [Dọc — O thắng]
  Bên đến lượt: O
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (5

'✓ PASS'

In [9]:
print('=== PHẦN 4B: Kiểm chứng chặn thua (block_urgent) — 4 hướng + biên ===')
print()

# Ngang — X chặn O (lượt X: 4 và 4)
b = make_board([(0, 0), (0, 2), (2, 0), (2, 2)], [(7, 3), (7, 4), (7, 5), (7, 6)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Ngang — X chặn O')

# Dọc — X chặn O
b = make_board([(5, 0), (5, 2), (7, 0), (7, 2)], [(0, 7), (1, 7), (2, 7), (3, 7)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Dọc — X chặn O')

# Chéo chính — X chặn O
b = make_board([(0, 2), (0, 4), (2, 0), (8, 8)], [(1, 1), (2, 2), (3, 3), (4, 4)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Chéo chính — X chặn O')

# Chéo phụ — X chặn O
b = make_board([(0, 0), (0, 2), (2, 0), (8, 8)], [(1, 9), (2, 8), (3, 7), (4, 6)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Chéo phụ — X chặn O')

# Biên dưới — O sát hàng 10
b = make_board([(5, 0), (5, 2), (7, 0), (7, 2)], [(10, 0), (10, 1), (10, 2), (10, 3)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Sát biên dưới — X chặn O')

# Biên trái — O sát cột 0
b = make_board([(5, 5), (5, 7), (7, 5), (7, 7)], [(0, 0), (1, 0), (2, 0), (3, 0)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Sát biên trái — X chặn O')

=== PHẦN 4B: Kiểm chứng chặn thua (block_urgent) — 4 hướng + biên ===

✓ PASS [Ngang — X chặn O]
  Bên đến lượt: X
  Luật kích hoạt: block_urgent (kỳ vọng: block_urgent)
  Nước chọn: (7, 7)
  Lý do: Chặn thua tại (7, 7) — đối thủ có 2 đe dọa thắng ngay, chặn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(7, 2), (7, 7)]
  Nước chọn hợp lệ: True

✓ PASS [Dọc — X chặn O]
  Bên đến lượt: X
  Luật kích hoạt: block_urgent (kỳ vọng: block_urgent)
  Nước chọn: (4, 7)
  Lý do: Chặn thua tại (4, 7) — đối thủ có 1 đe dọa thắng ngay, chặn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(4, 7)]
  Nước chọn hợp lệ: True

✓ PASS [Chéo chính — X chặn O]
  Bên đến lượt: X
  Luật kích hoạt: block_urgent (kỳ vọng: block_urgent)
  Nước chọn: (5, 5)
  Lý do: Chặn thua tại (5, 5) — đối thủ có 2 đe dọa thắng ngay, chặn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(0, 0), (5, 5)]
  Nước chọn hợp lệ: True

✓ PASS [Chéo phụ — X chặn O]
  Bên đế

'✓ PASS'

In [10]:
print('=== PHẦN 4C: Chuỗi đặc biệt — >5 quân và bị chặn 2 đầu ===')
print()

# Chuỗi >5 quân: X tại (0,1)(0,2)(0,3)(0,5)(0,6), ô (0,4) trống → nối 6 (lượt X: 5 và 5)
b = make_board([(0, 1), (0, 2), (0, 3), (0, 5), (0, 6)],
               [(2, 0), (2, 2), (4, 0), (4, 2), (6, 0)])
print(f'is_winning_move(board, 0, 4, X) = {is_winning_move(b, 0, 4, PLAYER_X)}')
test_scenario(b, PLAYER_X, 'win_now', [(0, 4)], 'Chuỗi 6 quân — X nối ở giữa')

# Chuỗi bị chặn 2 đầu: O X X . X X O, đặt X vào giữa → 5 quân liên tiếp (lượt X: 4 và 4)
b = make_board([(0, 2), (0, 3), (0, 5), (0, 6)], [(0, 1), (0, 7), (2, 0), (2, 2)])
print(f'is_winning_move(board, 0, 4, X) = {is_winning_move(b, 0, 4, PLAYER_X)}')
test_scenario(b, PLAYER_X, 'win_now', [(0, 4)], 'Chuỗi 5 bị chặn 2 đầu — X điền vào giữa')

=== PHẦN 4C: Chuỗi đặc biệt — >5 quân và bị chặn 2 đầu ===

is_winning_move(board, 0, 4, X) = True
✓ PASS [Chuỗi 6 quân — X nối ở giữa]
  Bên đến lượt: X
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (0, 4)
  Lý do: Thắng ngay tại (0, 4) — tìm thấy 1 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(0, 4)]
  Nước chọn hợp lệ: True

is_winning_move(board, 0, 4, X) = True
✓ PASS [Chuỗi 5 bị chặn 2 đầu — X điền vào giữa]
  Bên đến lượt: X
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (0, 4)
  Lý do: Thắng ngay tại (0, 4) — tìm thấy 1 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(0, 4)]
  Nước chọn hợp lệ: True



'✓ PASS'

In [11]:
print('=== PHẦN 4D: Xung đột ưu tiên — thắng ngay > chặn thua ===')
print()

# X có 4 quân ngang VÀ O có 4 quân dọc → X phải chọn thắng ngay (lượt X: 4 và 4)
b = make_board(
    [(5, 1), (5, 2), (5, 3), (5, 4)],
    [(0, 9), (1, 9), (2, 9), (3, 9)],
)
x_wins = find_winning_actions(b, PLAYER_X)
o_threats = find_winning_actions(b, PLAYER_O)
print(f'Nước thắng ngay của X: {x_wins}')
print(f'Đe dọa của O: {o_threats}')
test_scenario(b, PLAYER_X, 'win_now', x_wins, 'Xung đột: X thắng > chặn O')

# O có 4 quân dọc VÀ X có 4 quân ngang → O phải chọn thắng ngay (lượt O: 5 và 4)
b = make_board(
    [(8, 2), (8, 3), (8, 4), (8, 5), (10, 10)],
    [(0, 3), (1, 3), (2, 3), (3, 3)],
)
o_wins = find_winning_actions(b, PLAYER_O)
x_threats = find_winning_actions(b, PLAYER_X)
print(f'Nước thắng ngay của O: {o_wins}')
print(f'Đe dọa của X: {x_threats}')
test_scenario(b, PLAYER_O, 'win_now', o_wins, 'Xung đột: O thắng > chặn X')

=== PHẦN 4D: Xung đột ưu tiên — thắng ngay > chặn thua ===

Nước thắng ngay của X: [(5, 0), (5, 5)]
Đe dọa của O: [(4, 9)]
✓ PASS [Xung đột: X thắng > chặn O]
  Bên đến lượt: X
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (5, 5)
  Lý do: Thắng ngay tại (5, 5) — tìm thấy 2 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(5, 0), (5, 5)]
  Nước chọn hợp lệ: True

Nước thắng ngay của O: [(4, 3)]
Đe dọa của X: [(8, 1), (8, 6)]
✓ PASS [Xung đột: O thắng > chặn X]
  Bên đến lượt: O
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (4, 3)
  Lý do: Thắng ngay tại (4, 3) — tìm thấy 1 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(4, 3)]
  Nước chọn hợp lệ: True



'✓ PASS'

In [12]:
print('=== PHẦN 4E: Trường hợp biên — bàn trống và bàn gần đầy ===')
print()

# Bàn cờ trống (dựng qua CaroGame, X đi nước đầu)
b_empty = make_board([], [])
test_scenario(b_empty, PLAYER_X, 'positional_fallback', [(5, 5)], 'Bàn trống — X đi nước đầu')

# Bàn cờ chỉ còn 1 ô trống.
# GHI CHÚ: đây là trạng thái biên GIẢ ĐỊNH, dựng trực tiếp bằng numpy (toàn bộ ô là X
# trừ (10, 10)); không phải thế cờ hợp lệ theo luật engine, chỉ để kiểm tra hợp đồng
# "luôn trả về ô trống duy nhất".
b_one = np.ones((11, 11), dtype=np.int8)
b_one[10, 10] = EMPTY
print('Ô trống duy nhất: (10, 10)')
_agent_one = RuleBasedAgent()
action_one = _agent_one.choose_action(b_one, PLAYER_X)
print(f'Nước chọn: {action_one} (kỳ vọng: (10, 10))')
print(f'Kết quả: {"✓ PASS" if action_one == (10, 10) else "✗ FAIL"}')

=== PHẦN 4E: Trường hợp biên — bàn trống và bàn gần đầy ===



✓ PASS [Bàn trống — X đi nước đầu]
  Bên đến lượt: X
  Luật kích hoạt: positional_fallback (kỳ vọng: positional_fallback)
  Nước chọn: (5, 5)
  Lý do: Dự phòng vị trí tại (5, 5) — không có đe dọa trực tiếp, chọn từ 1 ứng viên bán kính 2 theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(5, 5)]
  Nước chọn hợp lệ: True

Ô trống duy nhất: (10, 10)
Nước chọn: (10, 10) (kỳ vọng: (10, 10))
Kết quả: ✓ PASS


In [13]:
print('=== PHẦN 4F: Thống kê nodes_visited và thinking_time ===')
print()

_agent_stats = RuleBasedAgent()
# Lượt X: |X| = |O| = 4
b_test = make_board([(5, 1), (5, 2), (5, 3), (5, 4)], [(0, 0), (0, 2), (0, 4), (0, 6)])

action = _agent_stats.choose_action(b_test, PLAYER_X)
print(f'Nước chọn: {action}')
print(f'nodes_visited: {_agent_stats.stats["nodes_visited"]}')
print(f'thinking_time: {_agent_stats.stats["thinking_time"]:.6f} giây')
print(f'Thông tin quyết định: {_agent_stats.explain_last_action()}')

=== PHẦN 4F: Thống kê nodes_visited và thinking_time ===

Nước chọn: (5, 5)
nodes_visited: 113
thinking_time: 0.000557 giây
Thông tin quyết định: {'rule_triggered': 'win_now', 'selected_action': (5, 5), 'candidate_count': 2, 'matching_actions': [(5, 0), (5, 5)], 'reason': 'Thắng ngay tại (5, 5) — tìm thấy 2 nước thắng, chọn theo tie-breaking.'}


### Nhận xét và phân tích kết quả Phần 4

Các nhận xét dưới đây dựa trên output thực tế của lần chạy ngay phía trên (18 ca `✓ PASS`, 0 ca `✗ FAIL`, không cell nào báo lỗi).

**1. Tính đúng đắn của luật thắng ngay và chặn thua**
- Toàn bộ các tình huống `win_now` (Phần 4A) ở cả 4 hướng (ngang, dọc, chéo chính, chéo phụ) và cả 2 phía X/O đều kích hoạt đúng luật kỳ vọng, nước chọn nằm trong tập `find_winning_actions`.
- Toàn bộ các tình huống `block_urgent` (Phần 4B) ở 4 hướng, kể cả sát biên dưới và sát biên trái, đều kích hoạt đúng luật và nước chọn nằm trong tập đe dọa của đối thủ.
- Các chuỗi đặc biệt (Phần 4C) được nhận diện đúng: nối thành chuỗi 6 quân tại `(0, 4)` và chuỗi 5 quân bị chặn cả 2 đầu vẫn được tính là thắng (`is_winning_move = True`), phù hợp luật của engine.
- Ở mọi ca, cờ `Board bất biến: True` — `get_action` không sửa mảng `board` đầu vào.

**2. Ưu tiên tuyệt đối của `win_now` so với `block_urgent`**
- Ở Phần 4D, X có nước thắng `[(5, 0), (5, 5)]` trong khi O đồng thời đe dọa tại `(4, 9)`: agent vẫn chọn `win_now` tại `(5, 5)` thay vì chặn O. Tình huống đảo bên (O thắng tại `(4, 3)` dù X đe dọa `(8, 1)`, `(8, 6)`) cho kết quả đối xứng.
- Lý do cấu trúc: `get_action` chỉ xuống tầng `block_urgent` khi tầng `win_now` không tìm được nước thắng nào, nên thắng ngay luôn được xét trước chặn thua. Điều này đúng về mặt lý thuyết vì thắng ngay kết thúc ván trước khi đối thủ kịp đi.

**3. Giới hạn quan sát được ở `block_urgent`**
- Khi đối thủ có chuỗi 4 quân hở cả hai đầu (ví dụ ca ngang `[(7, 2), (7, 7)]`, chéo chính `[(0, 0), (5, 5)]`, chéo phụ `[(0, 10), (5, 5)]`), agent chỉ chặn được một đầu; đối thủ vẫn thắng ở đầu còn lại ở lượt sau. Đây là giới hạn tự nhiên của luật 1 nước (đe dọa kép), không phải lỗi cài đặt, và là lý do cần nhận diện pattern sớm hơn (open-three) ở Giai đoạn 2.
- Các ca chỉ có một đầu bị đe dọa (dọc tại `(4, 7)`, sát biên dưới `(10, 4)`, sát biên trái `(4, 0)`) được chặn trọn vẹn.

**4. Chỉ số `nodes_visited` và `thinking_time` (Phần 4F)**
- Với thế X có 4 quân ngang, agent kích hoạt `win_now` ngay ở tầng 1 nên `nodes_visited = 113`, đúng bằng số ô trống (`121 - 8 = 113`) theo phương pháp đếm ở Phần 2; tầng `block_urgent` và `positional_fallback` không chạy nên không cộng thêm.
- `thinking_time` cỡ nửa mili-giây (khoảng 0.0005 giây), thấp hơn rất nhiều so với mốc hợp tác 1.9 giây; chi phí chủ yếu là một lượt quét toàn bàn (121 ô) với phép đếm 4 hướng cho mỗi ô trống.
- Lưu ý: giá trị `thinking_time` phụ thuộc máy và thay đổi nhẹ giữa các lần chạy; `nodes_visited` thì tất định.

---

## Phần 5 — Định nghĩa pattern và độ ưu tiên

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 2.

Nội dung dự kiến:
- Định nghĩa các pattern: open-four, open-three, chuỗi có khoảng trống, chuỗi bị chặn một đầu.
- Bảng điểm trọng số và thứ tự ưu tiên khi nhiều pattern cùng kích hoạt.
- Ví dụ đối chứng: nhận diện đúng vs. nhận diện nhầm (false positive).


## Phần 6 — Giải thích quyết định nước đi

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 2.

Nội dung dự kiến:
- Trực quan hóa luật được kích hoạt tại từng vị trí cờ.
- Phân tích các tình huống xung đột luật phức tạp.
- Dùng `explain_last_action()` để trace lý do chọn nước.


## Phần 7 — Đánh giá tác dụng thành phần (Ablation)

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 3.

Nội dung dự kiến:
- So sánh Full Pattern (`use_patterns=True`) vs. No Pattern (`use_patterns=False`).
- Giữ nguyên luật thắng/chặn ở cả hai biến thể.
- Tập vị trí kiểm tra, seed và cơ chế phá hòa đồng nhất.


## Phần 8 — Thi đấu thử nghiệm

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 3.

Nội dung dự kiến:
- Đấu với `RandomAgent`, đổi bên X/O, nhiều seed.
- Ghi nhận W/D/L, điểm số và `elapsed_s` trung bình.
- Dùng `evaluation/metrics.py` để tổng hợp kết quả.


## Phần 9 — Phân tích các trường hợp thất bại

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 3.

Nội dung dự kiến:
- Phân tích thế cờ đối phương tạo đe dọa kép (double threat).
- Nhận diện nhầm pattern hoặc giới hạn của tầng luật hiện tại.
- Ghi nhận như hạn chế kỹ thuật, không yêu cầu giải quyết bài toán bất khả thi.


## Phần 10 — Tổng kết và hướng dẫn tái lập

> **Trạng thái:** Phần tổng kết Giai đoạn 1 bên dưới; phần số liệu đầy đủ sẽ cập nhật ở Giai đoạn 3.

### Tổng kết Giai đoạn 1

**Câu trả lời tạm thời (chưa có số liệu thi đấu chính thức):**  
Luật thắng ngay và chặn thua hoạt động chính xác ở cả 4 hướng và 2 phía (X/O) theo kết quả kiểm thử tự động. Nước đi dự phòng vị trí ưu tiên tâm bàn cờ hợp lý cho giai đoạn đầu ván cờ.

**Hạn chế kỹ thuật đã biết:**
- Fallback chưa nhận diện các hình thái chiến thuật cơ bản (open-three, open-four).
- Đe dọa kép (double threat) không thể phòng thủ hoàn toàn trong 1 nước.
- Bán kính ứng viên fallback = 2 có thể bỏ sót vị trí chiến lược xa trung tâm quân cờ.

**Cấu hình đề xuất bàn giao:**  
`RuleBasedAgent(candidate_radius=2, use_patterns=False, seed=None)`

**Thứ tự chạy lại notebook:**  
Restart Kernel → Run All. Không phụ thuộc vào trạng thái biến từ cell trước.
